In [40]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder,OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('cleaned_dataset.csv')

In [41]:
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [42]:
X = df.drop(['Exam_Score'],axis=1)
y = df['Exam_Score']

In [43]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=10)

In [ ]:
numerical_features = ['Hours_Studied', 'Attendance','Previous_Scores']
ordinal_features = [
  'Motivation_Level',
  'Access_to_Resources',
  'Teacher_Quality',
  'Family_Income',
  'Parental_Involvement',
  'Peer_Influence',
  'Distance_from_Home',
  'Parental_Education_Level'
]
nominal_features = [
  'Gender',
  'School_Type',
  'Extracurricular_Activities',
  'Internet_Access',
  'Learning_Disabilities'
]
preprocessor = ColumnTransformer(
  [
    ('num',StandardScaler(),numerical_features),
    ('ord',OrdinalEncoder(),ordinal_features),
    ('nom',OneHotEncoder(),nominal_features)
  ],
  remainder='passthrough'
)
rl_pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('model',LinearRegression())
])

In [46]:
rl_pipeline.fit(X_train,y_train)
y_pred = rl_pipeline.predict(X_test)
r2 = r2_score(y_test,y_pred)
print(f'r2 score is: {round(r2*100,2)}%')

r2 score is: 60.58%


In [47]:
rl_pipeline.named_steps['model'].coef_

array([ 1.77203968,  2.28698357,  0.72594517, -0.15141651, -0.39676436,
       -0.21205131, -0.16472536, -0.44980428,  0.56246029,  0.50077639,
        0.20087237,  0.0097873 , -0.0097873 , -0.0055821 ,  0.0055821 ,
       -0.27879659,  0.27879659, -0.4782038 ,  0.4782038 ,  0.41681013,
       -0.41681013,  0.00452182,  0.48120755,  0.16882245])

In [48]:
rl_pipeline.named_steps['model'].intercept_

np.float64(65.30059823515776)

In [49]:
cv = KFold(n_splits=3,shuffle=True,random_state=10)
lr_CV = GridSearchCV(estimator=rl_pipeline,param_grid={'model__fit_intercept':[True,False],'model__positive':[True,False]},cv=cv,scoring='r2')
lr_CV.fit(X_train,y_train)
lr_CV.cv_results_
lr_CV.best_params_

{'model__fit_intercept': True, 'model__positive': False}

In [50]:
best_model = lr_CV.best_estimator_
y_pred = best_model.predict(X_test)

In [ ]:
r2 = r2_score(y_test,y_pred)
mse = mean_squared_error(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
rmse = np.sqrt(mse)
    print(f'mean squared error: {mse},\nmean absolute error: {mae},\nroot_mean_squared_error {rmse},\nand r2 score is: {round(r2*100,2)}%')

mean squared error: 5.8255314080155705,
mean absolute error: 1.097421299776286,
root_mean_squared_error 2.41361376529377,
and r2 score is: 60.58%
